In [1]:
import os
import subprocess

configs_dir = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/"
python = "/home/pfuerst/venvs/NNMFit0.5_py3.11/bin/python"
script = "/home/pfuerst/venvs/NNMFit0.5_py3.11/bin/calculate_graph.py"

In [2]:
main_out_dir = f"/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/step2_hese_flavor/graphs"
os.makedirs(main_out_dir, exist_ok=True)

In [7]:
model = "mcd-simpletopology_flux-hese_feat-11features_plus_rloglmilli_econf_evtgen"
threshold = "bdt1_0.333333_bdt2_0.366667_length_10"

In [13]:
DEFAULT_OVERRIDE_CONFIGS = [
    "override/livetime/hese_livetime_13yr_asr.cfg",
]
DEFAULT_OVERRIDE_COMPONENTS = [
    "override/components/astro_SPL_no_inel.yaml",
    "override/muon/muontemplate_hese_11features_plus_rloglmilli_econf_evtgen_bdt1_0.333333_bdt2_0.366667_paper.yaml",
]

DEFAULT_NO_SYSTEMATICS = ["override/systematics/NoSystematics_hese.cfg"]
DEFAULT_SYSTEMATICS = [f"override/systematics/hese/Snowstorm_Gradients_hese_HESEBestfit_SPL_with_flavor/{model}/{threshold}_wpriors.cfg"]

CONFIGS = []

In [14]:

CONFIGS += [
    {
        "name": f"all_param_2sigma_1D_10steps",
        "analysis_config": f"analysis_configs/data/hese/step2_hese_flavor/plotting/all_param_2sigma_1D_10steps/all_param_2sigma_1D_10steps.yaml",
        "main_config" : "main.cfg",
        "override_configs": [f"override/datasets_hese/data_v3/{model}/{threshold}.cfg",
                             f"override/binning/hese/10bdtprod_threshold_0.122.cfg"] + DEFAULT_OVERRIDE_CONFIGS,
        "override_components": DEFAULT_OVERRIDE_COMPONENTS,
        "override_parameters": None,
    },
]

In [10]:
unblind_script_path = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/"
import sys
sys.path.append(unblind_script_path)
from histogram_maker import *
from do_scan_analysis import *

In [15]:
for cfg in CONFIGS:
    name = cfg["name"]
    output_dir = os.path.join(main_out_dir, name)
    os.system(f"mkdir -p {output_dir}")
    outfile = os.path.join(output_dir,"Precalculated_Graph.pickle")

    print(10*"-")
    print(name)
    print(output_dir)
    print(outfile)

    NNMFit_config_options = {
        "init_method": "from_configs",
        "main_config": os.path.join(configs_dir, cfg["main_config"]),
        "analysis_config": os.path.join(configs_dir, cfg["analysis_config"]),
        "config_dir": configs_dir,
        "override_configs": cfg["override_configs"] + DEFAULT_SYSTEMATICS,
        "override_components": cfg["override_components"],
        "override_parameters": cfg["override_parameters"],
    }

    cmd = [
        python, script,
        "--main_config", os.path.join(configs_dir, cfg["main_config"]),
        "--analysis_config", os.path.join(configs_dir, cfg["analysis_config"]),
        "--config_dir", configs_dir,
        "--override_configs", *cfg["override_configs"] + DEFAULT_SYSTEMATICS,
        "--override_components", *cfg["override_components"],
        "-o", outfile,
    ]
    if cfg["override_parameters"]:
        cmd += ["--override_parameters", *cfg["override_parameters"]]

    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)

    # Total histogram WITH Snowstorm gradients (systematics)
    make_histogram_from_variables(NNMFit_config_options=NNMFit_config_options, out_file=f"{output_dir}/MC_Histogram.pickle")
    generate_data_hist(scan_path=output_dir)

    # Total histogram WITHOUT Snowstorm gradients — used to compute per-bin gradient at plot time
    NNMFit_config_options_nosyst = NNMFit_config_options.copy()
    NNMFit_config_options_nosyst["override_configs"] = cfg["override_configs"] + DEFAULT_NO_SYSTEMATICS
    make_histogram_from_variables(NNMFit_config_options=NNMFit_config_options_nosyst, out_file=f"{output_dir}/MC_Histogram_nosyst.pickle")

    # Per-component histograms WITHOUT Snowstorm gradients.
    # The per-bin gradient (total_syst - total_nosyst) is distributed among non-Muon
    # components at plot time weighted by component rate per bin (see plot_functions.py).
    for component in ["Astro", "Conv", "Muon"]:
        analysis_config_comp = os.path.join(configs_dir, cfg["analysis_config"].replace(".yaml", f"_{component}.yaml"))
        NNMFit_config_options_nosyst["analysis_config"] = analysis_config_comp
        print(10*"-", "Component (no syst)", component)
        print(analysis_config_comp)
        make_histogram_from_variables(NNMFit_config_options=NNMFit_config_options_nosyst, out_file=f"{output_dir}/MC_Histogram_{component}_nosyst.pickle")


----------
all_param_2sigma_1D_10steps
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/step2_hese_flavor/graphs/all_param_2sigma_1D_10steps
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/step2_hese_flavor/graphs/all_param_2sigma_1D_10steps/Precalculated_Graph.pickle

Called with:
main_config             : /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/main.cfg
analysis_config         : /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/step2_hese_flavor/plotting/all_param_2sigma_1D_10steps/all_param_2sigma_1D_10steps.yaml
config_dir              : /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/
override_configs        : ['override/datasets_hese/data_v3/mcd-simpletopology_flux-hese_feat-11features_plus_rloglmilli_econf_evtgen/bdt1_0.333333_bdt2_0.366667_length_10.cfg', 'override/binning/hese/10bdtprod_threshold_0.122.cfg', 'ov